#Fuentes de dato
Fuente  Sales Analysis on Northwind Database

https://www.kaggle.com/datasets/emmanueltugbeh/northwind-orders-and-order-details

Northwind base de datos de ejemplo creada por Microsoft.

Tablas utilizadas: **orders** (pedidos) y **order_details** (línea de pedido), relacionadas por **order_id**.

#Problema y objetivo

##Problema:

Los registros de pedidos de Northwind presentan errores de captura y formato que impiden calcular métricas de ventas confiables.

##Objetivo:

Diagnosticar la calidad de los datos como paso previo a la limpieza y el análisis.

# Extracción incial

##Dimensiones, columnas y tipo de tabla.

In [1]:
import gdown
id_details = "1xgyipusoHkOZEtYKIIJfqhVw1aIsEkIS"
archivo_details = f"https://drive.google.com/uc?id={id_details}"
details = "order_details.csv"

id_orders = "1WumjkcK1WaNtjWcLaLbPQNH5URJ3vSs0"
archivo_orders = f"https://drive.google.com/uc?id={id_orders}"
orders = "orders.csv"

gdown.download(archivo_details, details, quiet=False)
gdown.download(archivo_orders, orders, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1xgyipusoHkOZEtYKIIJfqhVw1aIsEkIS
To: /content/order_details.csv
100%|██████████| 41.7k/41.7k [00:00<00:00, 46.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WumjkcK1WaNtjWcLaLbPQNH5URJ3vSs0
To: /content/orders.csv
100%|██████████| 103k/103k [00:00<00:00, 56.2MB/s]


'orders.csv'

In [2]:
import pandas as pd

df_details = pd.read_csv(details, dtype=str, keep_default_na=False)
df_orders = pd.read_csv(orders, dtype=str, keep_default_na=False)

print("Order_details", df_details.shape)
print("Orders", df_orders.shape)
display(df_details.head())
display(df_orders.head())

Order_details (2215, 5)
Orders (865, 14)


,order_id,product_id,unit_price,quantity,discount
0,10474,40,14.7,21,0
1,10527,36,19,30,0.1
2,10731,51,53,30,0.05
3,11009,24,4.5,12,0
4,10657,60,34,30,0


,order_id,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
0,10869,SEVES,5,1998-02-04,1998-03-04,1998-02-09,1,143.28,Seven Seas Imports,90 Wadhurst Rd.,London,,OX15 4NB,UK
1,10440,SAVEA,4,1997-02-10,1997-03-10,1997-02-28,2,86.53,Save-a-lot Markets,187 Suffolk Ln.,Boise,ID,83720,USA
2,10951,RICSU,9,1998-03-16,1998-04-27,1998-04-07,2,30.85,Richter Supermarkt,Starenweg 5,Genève,,1204,Switzerland
3,10876,BONAP,7,1998-02-09,1998-03-09,1998-02-12,3,60.42,Bon app',"12, rue des Bouchers",Marseille,,13008,France
4,11047,EASTC,7,1998-04-24,1998-05-22,1998-05-01,3,46.62,Eastern Connection,35 King George,London,,WX3 6FW,U.K.


#Diagnóstico

##Valores Faltantes

##¿Por que pasar los datos como texto y no en el formato en que estaban?

Al pasar los datos como texto, se evita que pandas adivine los tipos y transforme la información. Por ende se quiere evitar esto para dejar evidencia de los errores del dataset.

Para diagnosticar la calidad de los datos es necesrio ver los valores tal como vienen en el archivo. Este es un primer paso del diagnóstico. Después, en la limpieza, sí se convertirá cada columna a su tipo correcto (númeor, fechas) con reglas claras y documentadas.


In [3]:
normal = pd.read_csv(details)

texto = df_details
print("LECTURA NORMAL")
print("Nulos (NaN) en unit_price:", normal["unit_price"].isna().sum())
print()
print("LECTURA COMO TEXTO")
print("Vacíos:", (texto["unit_price"].str.strip() == "").sum())
print("Valores 'N/A'", (texto["unit_price"] == "N/A").sum())
print("Valores 'free':", (texto["unit_price"] == "free").sum())

LECTURA NORMAL
Nulos (NaN) en unit_price: 33

LECTURA COMO TEXTO
Vacíos: 25
Valores 'N/A' 8
Valores 'free': 6


La lecutra normal reporta 33 valores nulos, pero en realidad son 25 vacíos más 8 **N/A** mezclados. No distingue entre una celda vacía y un **N/A** escrito, y los 6 **free** ni aparecen como problema.

###Conteo de valores faltantes

In [4]:
def faltantes(df):
  vacios = df.apply(lambda col: col.str.strip().eq("").sum())
  resumen = pd.DataFrame({
      "vacios": vacios,
      "porcentaje": (vacios / len(df) * 100).round(2)
  })
  return resumen.sort_values("vacios", ascending=False)


print("order_details")
display(faltantes(df_details))
print()
print("orders")
display(faltantes(df_orders))

order_details


,vacios,porcentaje
discount,31,1.40
unit_price,25,1.13
quantity,20,0.90
product_id,15,0.68
order_id,0,0.00



orders


,vacios,porcentaje
ship_region,525,60.69
freight,27,3.12
shipped_date,23,2.66
ship_postal_code,21,2.43
customer_id,16,1.85
ship_city,15,1.73
ship_name,12,1.39
employee_id,12,1.39
order_date,11,1.27
required_date,11,1.27


Se recorre cada columna del DataFrame, se identifica las celdas que están vacías o contienen únicamente espacios y cuenta cuántas hay en cada columna.

###Registros duplicados

Se revisan en esta sección dos tipos de duplicados: filas idénticas en todas sus columnas y filas que repiten la clave del registro (order_id en orders; order_id + product_id en order_details) aunque difieran en algún otro campo.

In [5]:


exactos = df_orders.duplicated().sum()
por_id = df_orders.duplicated(subset="order_id").sum()

print("orders")
print("Filas totales:", len(df_orders))
print("order_id únicos:", df_orders["order_id"].nunique())
print("Filas idénticas repetidas:", exactos)
print("Filas con order_id repetido (incluye las idénticas):", por_id)
print("Repetidas con order_id igual pero contenido distinto", por_id - exactos)


orders
Filas totales: 865
order_id únicos: 830
Filas idénticas repetidas: 25
Filas con order_id repetido (incluye las idénticas): 35
Repetidas con order_id igual pero contenido distinto 10


In [6]:
sin_extractos = df_orders.drop_duplicates()
conflicto = sin_extractos[sin_extractos.duplicated(subset="order_id", keep=False)]
conflicto = conflicto.sort_values("order_id")

print("Filas en conflicto:", len(conflicto), "| pedidos afectados:", conflicto["order_id"].nunique())
display(conflicto.head(4))

Filas en conflicto: 20 | pedidos afectados: 10


,order_id,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
620,10263,ERNSH,9,1996-07-23,1996-08-20,1996-07-31,3,146.06,ERNST HANDEL,Kirchgasse 6,Graz,,8010,Austria
452,10263,ERNSH,9,1996-07-23,1996-08-20,1996-07-31,3,146.06,Ernst Handel,Kirchgasse 6,Graz,,8010,Austria
127,10349,SPLIR,7,1996-11-08,1996-12-06,1996-11-15,1,8.63,Split Rail Beer & Ale,P.O. Box 555,LaÃ¼nder,WY,82520,USA
728,10349,SPLIR,7,1996-11-08,1996-12-06,1996-11-15,1,8.63,SPLIT RAIL BEER & ALE,P.O. Box 555,LaÃ¼nder,WY,82520,USA


In [7]:
display(conflicto[["order_id", "customer_id", "order_date", "ship_name"]])

,order_id,customer_id,order_date,ship_name
620,10263,ERNSH,1996-07-23,ERNST HANDEL
452,10263,ERNSH,1996-07-23,Ernst Handel
127,10349,SPLIR,1996-11-08,Split Rail Beer & Ale
728,10349,SPLIR,1996-11-08,SPLIT RAIL BEER & ALE
36,10412,WARTH,1997-01-13,WARTIAN HERKKU
71,10412,WARTH,1997-01-13,Wartian Herkku
55,10457,KOENE,1997-02-25,KÖNIGLICH ESSEN
82,10457,KOENE,1997-02-25,Königlich Essen
260,10552,HILAA,1997-05-29,HILARION-Abastos
515,10552,HILAA,1997-05-29,HILARION-ABASTOS


In [8]:
difieren = conflicto.groupby("order_id").agg(lambda s: s.nunique() > 1).any()
print("Columnas que difieren entre las copias:", difieren[difieren].index.tolist())
mismo_nombre = conflicto.groupby("order_id")["ship_name"].apply(lambda s: s.str.lower().nunique() == 1)
print("Pedidos donde ship_name solo cambia en mayúsculas:", mismo_nombre.sum(), "de", len(mismo_nombre))

Columnas que difieren entre las copias: ['ship_name']
Pedidos donde ship_name solo cambia en mayúsculas: 10 de 10


In [9]:
exactos_d = df_details.duplicated().sum()
por_clave = df_details.duplicated(subset=["order_id", "product_id"]).sum()
sin_exactos_d = df_details.drop_duplicates()
por_clave_tras = sin_exactos_d.duplicated(subset=["order_id", "product_id"]).sum()

print("order_details")
print("Filas totales:", len(df_details))
print("Filas idénticas repetidas:", exactos_d)
print("Repetidas por order_id + product_id:", por_clave)
print("Repetidas por clave tras quitar las idénticas:", por_clave_tras)

order_details
Filas totales: 2215
Filas idénticas repetidas: 60
Repetidas por order_id + product_id: 60
Repetidas por clave tras quitar las idénticas: 0


**Conclusion:** En orders se encontraron 25 filas identicas repetidas y 10 pedidos registrados dos veces con contenido distinto; en todos esos casos la unica diferencia esta en ship_name, escrito con mayusculas distintas.

En order_details se encontraron 60 filas identicas repetidas y ninun conflicto por clave. Tras eliminar los duplicados exactos, quedarian 840 filas en orders (todavia con 10 pedidos repetidos por el formato de ship_name) y 2.155 en order_details.

###Tipos de dato incorrectos
Según el diccionario de datos, order_id, product_id y quantity deben ser enteros, y
unit_price y discount deben ser decimales. Como los datos se cargaron como texto, se
intenta convertir cada columna a número y se listan los valores que no se pueden
convertir (sin contar los vacíos, que ya se cuantificaron en la sección de faltantes).


In [10]:
def revisar_numerica(df, columna, entero=False):
  serie = df[columna].str.strip()
  vacios = (serie == "").sum()
  convertido = pd.to_numeric(serie, errors="coerce")
  no_numericos = serie[convertido.isna() & (serie != "")]

  print(f"--- {columna} ---")
  print("Convertibles a número:", convertido.notna().sum())
  print("Vacíos:", vacios)
  print("No numéricos:", len(no_numericos))
  if len(no_numericos) > 0:
    display(no_numericos.value_counts())

  if entero:
    con_decimales = serie[convertido.notna() & (convertido % 1 != 0)]
    print("Con decimales (debería ser entero):", len(con_decimales))
    if len(con_decimales) > 0:
      print(con_decimales.tolist())
  print()



In [11]:
revisar_numerica(df_details, "order_id", entero=True)
revisar_numerica(df_details, "product_id", entero=True)
revisar_numerica(df_details, "unit_price")
revisar_numerica(df_details, "quantity", entero=True)
revisar_numerica(df_details, "discount")

--- order_id ---
Convertibles a número: 2214
Vacíos: 0
No numéricos: 1


order_id
A10363    1
Name: count, dtype: int64

Con decimales (debería ser entero): 0

--- product_id ---
Convertibles a número: 2200
Vacíos: 15
No numéricos: 0
Con decimales (debería ser entero): 0

--- unit_price ---
Convertibles a número: 2162
Vacíos: 25
No numéricos: 28


unit_price
N/A          8
free         6
21.05 USD    1
$12          1
30,4         1
123,79       1
$27.8        1
$30.4        1
7.75 USD     1
50 USD       1
$7.7         1
15.5 USD     1
$50          1
$27.2        1
$10          1
25.89 USD    1
Name: count, dtype: int64


--- quantity ---
Convertibles a número: 2192
Vacíos: 20
No numéricos: 3


quantity
diez    3
Name: count, dtype: int64

Con decimales (debería ser entero): 3
['42.5', '2.5', '15.5']

--- discount ---
Convertibles a número: 2175
Vacíos: 31
No numéricos: 9


discount
0,05     4
15%      3
cinco    2
Name: count, dtype: int64

In [12]:
revisar_numerica(df_orders, "order_id", entero=True)
revisar_numerica(df_orders, "employee_id", entero=True)
revisar_numerica(df_orders, "ship_via", entero=True)
revisar_numerica(df_orders, "freight")

--- order_id ---
Convertibles a número: 865
Vacíos: 0
No numéricos: 0
Con decimales (debería ser entero): 0

--- employee_id ---
Convertibles a número: 853
Vacíos: 12
No numéricos: 0
Con decimales (debería ser entero): 0

--- ship_via ---
Convertibles a número: 865
Vacíos: 0
No numéricos: 0
Con decimales (debería ser entero): 0

--- freight ---
Convertibles a número: 822
Vacíos: 27
No numéricos: 16


freight
$89           1
176,48        1
9,26          1
$35.03        1
110,37        1
$4.2          1
45,13         1
38,06         1
22,21         1
151.52 USD    1
64.5 USD      1
47,22         1
38,82         1
141.06 USD    1
$7.13         1
17.68 USD     1
Name: count, dtype: int64

In [13]:
for col in ["order_date", "required_date", "shipped_date"]:
  s = df_orders[col].str.strip()
  iso = s.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
  print(f"--- {col} ---")
  print("Con formato AAAA-MM-DD:", iso.sum())
  print("Vacíos:", (s == "").sum())
  print("Con otro formato:", (~iso & (s !="")).sum())
  print()

--- order_date ---
Con formato AAAA-MM-DD: 790
Vacíos: 11
Con otro formato: 64

--- required_date ---
Con formato AAAA-MM-DD: 794
Vacíos: 11
Con otro formato: 60

--- shipped_date ---
Con formato AAAA-MM-DD: 781
Vacíos: 23
Con otro formato: 61



**Conclusión:** En order_details se encontraron 41 valores que no se pueden convertir al tipo esperado: 28 en unit_price (N/A, free, símbolos de moneda y comas decimales), 9 en discount (comas decimales, porcentajes y palabras), 3 en quantity (texto) y 1 en order_id(A10363); además, 3 cantidades tienen decimales. En orders, freight tiene 16 valores no numéricos (comas decimales y símbolos de moneda) y las tres columnas de fecha tienen entre 60 y 64 valores que no siguen el formato AAAA-MM-DD. Las columnas emplyee_id y ship_via no presentan problemas de tipo, y order_id de orders tampoco.

###Formatos inconsistentes

Se revisan los valores que representan lo mismo pero están escritos de distinta forma: espacios sobrantes, mayúsculas y minúsculas, nombres de país y formatos de fecha.

In [14]:
for nombre, df in [("orders", df_orders), ("order_details", df_details)]:
    print(nombre)
    for col in df.columns:
        n = (df[col] != df[col].str.strip()).sum()
        if n > 0:
            print(f"  {col}: {n} valores con espacios al inicio o al final")
    print()

print("order_id con '.0' al final en order_details:",
      df_details["order_id"].str.strip().str.endswith(".0").sum())

orders
  customer_id: 26 valores con espacios al inicio o al final
  ship_name: 27 valores con espacios al inicio o al final
  ship_city: 26 valores con espacios al inicio o al final

order_details
  order_id: 6 valores con espacios al inicio o al final

order_id con '.0' al final en order_details: 3


In [15]:
def variantes_mayusculas(df, columna):
  s = df[columna].str.strip()
  s = s[s != ""]
  return (s.groupby(s.str.lower()).nunique() > 1).sum()

for col in ["customer_id", "ship_name", "ship_address", "ship_city", "ship_region", "ship_country"]:
  print(col, "-> valores escritos de más de una forma:", variantes_mayusculas(df_orders, col))


customer_id -> valores escritos de más de una forma: 0
ship_name -> valores escritos de más de una forma: 21
ship_address -> valores escritos de más de una forma: 8
ship_city -> valores escritos de más de una forma: 14
ship_region -> valores escritos de más de una forma: 0
ship_country -> valores escritos de más de una forma: 10


In [16]:
paises_validos = ["USA", "Germany", "Brazil", "France", "UK", "Venezuela", "Austria",
                  "Sweden", "Italy", "Canada", "Mexico", "Finland", "Ireland", "Spain",
                  "Denmark", "Switzerland", "Belgium", "Argentina", "Portugal", "Poland", "Norway"]

pais = df_orders["ship_country"].str.strip()
no_estandar = pais[(pais != "") & ~pais.isin(paises_validos)]

print("Valores distintos (incluye el vacío):", pais.nunique())
print("Filas con país estándar:", pais.isin(paises_validos).sum())
print("Filas vacías:", (pais == "").sum())
print("Filas con país no estándar:", len(no_estandar), "| variantes distintas:", no_estandar.nunique())
display(no_estandar.value_counts())

Valores distintos (incluye el vacío): 50
Filas con país estándar: 789
Filas vacías: 10
Filas con país no estándar: 66 | variantes distintas: 28


ship_country
United States     6
france            5
GERMANY           5
Germnay           5
brazil            4
uk                3
Frnace            3
México            3
U.K.              2
Usa               2
Italia            2
Méxco             2
United Kingdom    2
Finalnd           2
EEUU              2
Germay            2
Francia           2
Brasil            2
spain             2
canada            2
sweden            1
austria           1
Venezuala         1
Itally            1
germany           1
Canda             1
venezuela         1
Austira           1
Name: count, dtype: int64

In [17]:
import re

nombres = {
    "9999-99-99": "AAAA-MM-DD (correcto)",
    "99/99/9999": "DD/MM/AAAA o MM/DD/AAAA",
    "99-99-9999": "DD-MM-AAAA o MM-DD-AAAA",
    "99.99.99": "DD.MM.AA",
    "99999999": "AAAAMMDD",
    "9999/99/99": "AAAA/MM/DD",
    "9 A A A 9999": "D de mes de AAAA",
    "99 A A A 9999": "D de mes de AAAA",
    "A 99, 9999": "Mes DD, AAAA",
}

def formato_fecha(x):
    x = x.strip()
    if x == "":
        return "vacío"
    patron = re.sub(r"[A-Za-zÁ-ú]+", "A", re.sub(r"\d", "9", x))
    return nombres.get(patron, "otro")

columnas_fecha = ["order_date", "required_date", "shipped_date"]
tabla = pd.DataFrame({c: df_orders[c].map(formato_fecha).value_counts() for c in columnas_fecha})
display(tabla.fillna(0).astype(int))

,order_date,required_date,shipped_date
AAAA-MM-DD (correcto),790,794,781
AAAA/MM/DD,9,3,13
AAAAMMDD,9,7,11
D de mes de AAAA,13,11,5
DD-MM-AAAA o MM-DD-AAAA,8,10,8
DD.MM.AA,10,3,11
DD/MM/AAAA o MM/DD/AAAA,10,11,9
"Mes DD, AAAA",5,15,4
vacío,11,11,23


In [18]:
total = ambiguas = dia_primero = mes_primero = 0
for col in columnas_fecha:
    s = df_orders[col].str.strip()
    m = s[s.str.fullmatch(r"\d{2}[/-]\d{2}[/-]\d{4}")]
    a = m.str[:2].astype(int)
    b = m.str[3:5].astype(int)
    total += len(m)
    ambiguas += ((a <= 12) & (b <= 12)).sum()
    dia_primero += (a > 12).sum()
    mes_primero += (b > 12).sum()

print("Fechas con barra o guion y año de 4 dígitos:", total)
print("Ambiguas (día y mes ≤ 12):", ambiguas)
print("Con el día primero (primer número > 12):", dia_primero)
print("Con el mes primero (segundo número > 12):", mes_primero)

Fechas con barra o guion y año de 4 dígitos: 56
Ambiguas (día y mes ≤ 12): 19
Con el día primero (primer número > 12): 20
Con el mes primero (segundo número > 12): 17


**Conclusión.** En orders se encontraron espacios sobrantes en customer_id (26),
ship_name (27) y ship_city (26), y en order_details en order_id (6, más 3 con ".0" al
final). Varios campos de texto se escriben de más de una forma por diferencias de
mayúsculas (ship_name 21, ship_city 14, ship_country 10, ship_address 8). En ship_country
hay 28 variantes no estándar (66 filas) frente a una lista de 21 países válidos:
mayúsculas, errores de escritura, otro idioma y siglas. Las tres columnas de fecha usan 8
formatos distintos; en las que traen barra o guion con año de 4 dígitos conviven las
convenciones día/mes y mes/día (20 y 17 casos), y 19 de ellas son ambiguas.

###Valores fuera de rango

Se revisan valores que tienen el tipo correcto pero que no son posibles según el diccionario de datos: cantidades no positivas, descuentos no permitidos, fletes negativos y fechas incoherentes entre sí.

In [19]:
def a_numero(serie):
    return pd.to_numeric(serie.str.strip(), errors="coerce")

q = a_numero(df_details["quantity"])
p = a_numero(df_details["unit_price"])
desc = a_numero(df_details["discount"])

print("quantity ≤ 0:", (q <= 0).sum(), sorted(q[q <= 0].tolist()))
print("quantity > 130:", (q > 130).sum(), q[q > 130].tolist())
print("unit_price ≤ 0:", (p <= 0).sum())

descuentos_validos = [0, 0.05, 0.10, 0.15, 0.20, 0.25]
desc_invalido = desc[desc.notna() & ~desc.isin(descuentos_validos)]
print("discount fuera de los valores permitidos:", len(desc_invalido))
display(desc_invalido.value_counts())

quantity ≤ 0: 5 [-25.0, -24.0, -5.0, 0.0, 0.0]
quantity > 130: 1 [999.0]
unit_price ≤ 0: 0
discount fuera de los valores permitidos: 15


discount
-0.10    3
 0.03    3
 1.50    3
 0.02    2
 0.04    2
 0.01    1
 0.06    1
Name: count, dtype: int64

In [20]:
f = a_numero(df_orders["freight"])
e = a_numero(df_orders["employee_id"])
v = a_numero(df_orders["ship_via"])

print("freight < 0:", (f < 0).sum(), sorted(f[f < 0].tolist()))
print("freight máximo:", f.max())
print("employee_id fuera de 1–9:", ((e < 1) | (e > 9)).sum())
print("ship_via fuera de {1, 2, 3}:", (~v.dropna().isin([1, 2, 3])).sum())

freight < 0: 4 [-136.54, -24.69, -12.36, -10.22]
freight máximo: 1007.64
employee_id fuera de 1–9: 0
ship_via fuera de {1, 2, 3}: 0


In [21]:
def fecha_iso(col):
    s = df_orders[col].str.strip()
    return pd.to_datetime(s.where(s.str.fullmatch(r"\d{4}-\d{2}-\d{2}")),
                          format="%Y-%m-%d", errors="coerce")

pedido, requerido, enviado = fecha_iso("order_date"), fecha_iso("required_date"), fecha_iso("shipped_date")

print("Rango de order_date:", pedido.min().date(), "a", pedido.max().date())

ok = pedido.notna() & enviado.notna()
print("Enviado antes de la fecha del pedido:", (enviado[ok] < pedido[ok]).sum(), "de", ok.sum())
ok = pedido.notna() & requerido.notna()
print("Requerido antes de la fecha del pedido:", (requerido[ok] < pedido[ok]).sum(), "de", ok.sum())
ok = requerido.notna() & enviado.notna()
print("Enviado después de la fecha requerida:", (enviado[ok] > requerido[ok]).sum(), "de", ok.sum())

Rango de order_date: 1996-07-04 a 1998-05-06
Enviado antes de la fecha del pedido: 0 de 713
Requerido antes de la fecha del pedido: 0 de 730
Enviado después de la fecha requerida: 35 de 713


**Conclusión.** En order_details se encontraron 5 cantidades no positivas (-25, -24, -5,
0 y 0), 1 cantidad atípica (999) y 15 descuentos fuera de los valores permitidos (-0.1,
0.01, 0.02, 0.03, 0.04, 0.06 y 1.5); ningún precio es menor o igual a cero. En orders hay
4 fletes negativos, y employee_id y ship_via están dentro de los rangos esperados. Entre
las fechas con formato AAAA-MM-DD no hay incoherencias de orden: ningún pedido se envió
ni se requirió antes de hacerse. Los 35 envíos posteriores a la fecha requerida son una
observación de negocio y no un error de calidad.

### Integridad entre tablas

Se comprueba que cada order_id de order_details exista en orders, y que cada pedido de
orders tenga al menos una línea en order_details. Primero se compara con los valores tal
como vienen y después quitando los espacios y el ".0" final.

In [22]:
ids_orders = set(df_orders["order_id"].str.strip())
ids_details = set(df_details["order_id"])

print("Pedidos distintos en orders:", len(ids_orders))
print("order_id distintos en order_details:", len(ids_details))
print("order_id de order_details que no existen en orders:", len(ids_details - ids_orders))
print("Pedidos de orders sin líneas en order_details:", len(ids_orders - ids_details),
      sorted(ids_orders - ids_details))

Pedidos distintos en orders: 830
order_id distintos en order_details: 842
order_id de order_details que no existen en orders: 15
Pedidos de orders sin líneas en order_details: 3 ['10422', '10545', '11061']


In [23]:
ids_norm = df_details["order_id"].str.strip().str.replace(r"\.0$", "", regex=True)
huerfanos = ids_norm[~ids_norm.isin(ids_orders)]

print("Tras quitar espacios y '.0':")
print("order_id de order_details sin pedido en orders:", huerfanos.nunique(), sorted(huerfanos.unique()))
print("Líneas afectadas:", len(huerfanos))
print("Pedidos de orders sin líneas:", len(ids_orders - set(ids_norm)))

Tras quitar espacios y '.0':
order_id de order_details sin pedido en orders: 6 ['19473', '19475', '19484', '19550', '19614', 'A10363']
Líneas afectadas: 6
Pedidos de orders sin líneas: 0


**Conclusión.** Al comparar los order_id tal como vienen, 15 de order_details no existen
en orders y 3 pedidos de orders aparecen sin líneas. La mayor parte se debe a 9
identificadores con espacios o ".0" al final: una vez normalizados, los 830 pedidos
tienen al menos una línea y quedan 6 líneas sin pedido (19473, 19475, 19484, 19550,
19614 y A10363). Parecen errores de captura de pedidos que sí existen (10473, 10475,
10484, 10550, 10614 y 10363), pero con los datos disponibles no se puede confirmar.

### Caracteres mal codificados

Se buscan en las columnas de texto caracteres que no pertenecen al texto original: las
secuencias "Ã" y "Â" que aparecen cuando una letra con acento se lee con la codificación
equivocada, y el carácter de reemplazo "�". Las letras con acento legítimas (é, ö, ñ...)
no se cuentan.

In [24]:
patron = r"[ÃÂ�]"

for nombre, df in [("orders", df_orders), ("order_details", df_details)]:
    total = 0
    print(nombre)
    for col in df.columns:
        n = df[col].str.contains(patron, regex=True).sum()
        if n > 0:
            total += n
            print(f"  {col}: {n} valores con caracteres extraños")
    print("  Total de celdas afectadas:", total)
    print()

orders
  ship_name: 20 valores con caracteres extraños
  ship_address: 8 valores con caracteres extraños
  ship_city: 11 valores con caracteres extraños
  ship_postal_code: 7 valores con caracteres extraños
  Total de celdas afectadas: 46

order_details
  Total de celdas afectadas: 0



In [25]:
cols_texto = ["ship_name", "ship_address", "ship_city", "ship_postal_code"]
afectadas = df_orders[df_orders[cols_texto].apply(lambda c: c.str.contains(patron, regex=True)).any(axis=1)]

print("Filas afectadas:", len(afectadas))
print("Con el carácter �:", sum(df_orders[c].str.contains("�").sum() for c in df_orders.columns))
print("Con secuencias tipo 'Ã¼', 'Ã±', 'Ã©':", sum(df_orders[c].str.contains("[ÃÂ]", regex=True).sum() for c in df_orders.columns))
display(afectadas[["order_id"] + cols_texto].head(10))

Filas afectadas: 46
Con el carácter �: 23
Con secuencias tipo 'Ã¼', 'Ã±', 'Ã©': 23


,order_id,ship_name,ship_address,ship_city,ship_postal_code
5,10915,Tortuga Restaurante,Avda. Azteca 123,México D.F.,05Ã¼033
17,10904,White Clover Markets,1029 - 12th Ave. S.,SeÃ¼attle,98124
75,10565,Mère Paillarde,43 rue St. Laurent,Montréal,H�1J 1C3
84,10328,Furia Bacalhau e Frutos do Mar,Jardim das rosas nÃ©. 32,,1675
86,10606,�Tradiçao Hipermercados,"Av. Inês de Castro, 414",Sao Paulo,05634-030
92,10443,Reggiani CaseificÃ©i,Strada Provinciale 124,Reggio Emilia,42100
118,10680,Old World Delicatessen,2743 Bering St.,Anc�horage,99508
127,10349,Split Rail Beer & Ale,P.O. Box 555,LaÃ¼nder,82520
188,10494,Comércio Mineiro,"Av. dos �Lusíadas, 23",Sao Paulo,05432-043
222,10973,La corne d'abondancÃ±e,"67, avenue de l''Europe",Versailles,78000


In [26]:
ejemplos = df_orders.loc[df_orders["ship_city"].str.contains("Ã", regex=False), "ship_city"].unique()
for x in ejemplos:
    print(x, "→", x.encode("latin-1").decode("utf-8"))

SeÃ¼attle → Seüattle
LaÃ¼nder → Laünder
TsawaÃ±ssen → Tsawañssen


**Conclusión.** En orders se encontraron 46 celdas con caracteres extraños, cada una en
una fila distinta: ship_name (20), ship_city (11), ship_address (8) y ship_postal_code
(7); order_details no tiene casos. La mitad contiene el carácter de reemplazo "�" y la
otra mitad secuencias como "Ã¼", "Ã±" o "Ã©". La reparación habitual de mojibake no
recupera el texto original (por ejemplo, "SeÃ¼attle" pasa a "Seüattle" y no a
"Seattle"), lo que sugiere que los caracteres fueron insertados dentro de las palabras y
que la limpieza deberá eliminarlos.

## Resumen general del diagnóstico

| Categoría | `orders` | `order_details` |
|---|---|---|
| **Valores faltantes** | `ship_region` 525 (esperables), `freight` 27, `shipped_date` 23, `ship_postal_code` 21, `customer_id` 16, entre otras | `discount` 31, `unit_price` 25, `quantity` 20, `product_id` 15 |
| **Duplicados** | 25 filas idénticas; 10 pedidos repetidos con `ship_name` en otras mayúsculas | 60 filas idénticas; 0 repetidas por `order_id` + `product_id` |
| **Tipos incorrectos** | `freight`: 16 no numéricos; fechas fuera de `AAAA-MM-DD`: 64 (`order_date`), 60 (`required_date`) y 61 (`shipped_date`) | `unit_price` 28, `discount` 9, `quantity` 3 (más 3 con decimales), `order_id` 1 |
| **Formatos inconsistentes** | 8 formatos de fecha (19 de las 56 fechas con barra o guion son ambiguas); 66 filas con país no estándar (28 variantes); espacios en `customer_id` (26), `ship_name` (27) y `ship_city` (26); mayúsculas inconsistentes en `ship_name`, `ship_city`, `ship_country` y `ship_address` | `order_id` con espacios (6) o con `.0` (3) |
| **Valores fuera de rango** | 4 fletes negativos | 5 cantidades no positivas, 1 atípica (999) y 15 descuentos no permitidos |
| **Integridad entre tablas** | 3 pedidos sin líneas (0 tras normalizar los `order_id`) | 15 `order_id` sin pedido (6 tras normalizar: 19473, 19475, 19484, 19550, 19614 y A10363) |
| **Encoding** | 46 celdas con caracteres insertados: `ship_name` 20, `ship_city` 11, `ship_address` 8, `ship_postal_code` 7 | 0 |

*Las cifras corresponden a los datos originales, antes de la limpieza, y un mismo registro puede aparecer en más de una categoría.*

# Avance 2

##Transformación de los datos

En esta parte se aplican las transformaciones a partir de lo encontrado en el diagnóstico. Se trabaja sobre copias de las tablas para no modificar los datos originales.

#Transformación 1

##Unicidad de clave (Control 1)

In [27]:
orders_dep = df_orders.drop_duplicates().copy()


orders_dep["_llenos"] = orders_dep.apply(lambda c: c.str.strip().ne("")).sum(axis=1)
orders_dep = (orders_dep
              .sort_values("_llenos", ascending=False, kind="stable")
              .drop_duplicates(subset="order_id", keep="first")
              .drop(columns="_llenos")
              .sort_values("order_id"))

In [28]:
print("Filas:", len(orders_dep))
print("order_id únicos:", orders_dep["order_id"].nunique())
print("Duplicados por order_id:", orders_dep.duplicated(subset="order_id").sum())
assert orders_dep["order_id"].is_unique

Filas: 830
order_id únicos: 830
Duplicados por order_id: 0


##Control 2: reconciliación de totales

In [29]:
def importe(df):
    p = pd.to_numeric(df["unit_price"], errors="coerce")
    q = pd.to_numeric(df["quantity"], errors="coerce")
    d = pd.to_numeric(df["discount"], errors="coerce").fillna(0)
    return p * q * (1 - d)

df_details["importe"] = importe(df_details)

total_lineas = df_details["importe"].sum()
con_orden = df_details["order_id"].isin(orders_dep["order_id"])
total_con_orden = df_details.loc[con_orden, "importe"].sum()

print("Líneas totales:", len(df_details))
print("Líneas sin orden en orders_dep:", (~con_orden).sum())
print("Importe total details:", round(total_lineas, 2))
print("Importe con orden válida:", round(total_con_orden, 2))
print("Diferencia:", round(total_lineas - total_con_orden, 2))
print("Líneas sin importe calculable (precio/cantidad vacíos):", df_details["importe"].isna().sum())

Líneas totales: 2215
Líneas sin orden en orders_dep: 15
Importe total details: 1254983.16
Importe con orden válida: 1250608.41
Diferencia: 4374.75
Líneas sin importe calculable (precio/cantidad vacíos): 76


In [30]:
huerfanas = df_details.loc[~con_orden].copy()
details_dep = df_details.loc[con_orden].copy()

print("Líneas:", len(df_details), "=", len(details_dep), "+", len(huerfanas))
print("Importe:", round(total_lineas, 2), "=",
      round(details_dep["importe"].sum(), 2), "+", round(huerfanas["importe"].sum(), 2))
print("Descuadre líneas:", len(df_details) - len(details_dep) - len(huerfanas))
print("Descuadre importe:", round(total_lineas - details_dep["importe"].sum() - huerfanas["importe"].sum(), 2))
assert len(details_dep) + len(huerfanas) == len(df_details)

Líneas: 2215 = 2200 + 15
Importe: 1254983.16 = 1250608.41 + 4374.75
Descuadre líneas: 0
Descuadre importe: 0.0


| Control | Problema / riesgo | Corrección o decisión | Evidencia de nueva prueba |
|---|---|---|---|
| Unicidad de clave (`orders.order_id`) | `orders` tenía 865 filas pero solo 830 `order_id` únicos. Había 35 filas repetidas por clave: 25 duplicados exactos y 10 con el mismo `order_id` y contenido distinto. El riesgo era duplicar pedidos al unir con `order_details` e inflar conteos e importes. | Se eliminaron los duplicados exactos con `drop_duplicates()`. Para los 10 casos con contenido distinto se decidió conservar, por cada `order_id`, la fila con más campos llenos. Resultado: `orders_dep`. | Al volver a ejecutar: 830 filas, 830 `order_id` únicos y 0 duplicados por `order_id`. La validación `assert orders_dep["order_id"].is_unique` pasó sin error. |
| Reconciliación de totales (`order_details` vs `orders_dep`) | De las 2215 líneas de `order_details`, 15 tenían un `order_id` que no existe en `orders_dep`, y aportaban 4,374.75 al importe total de 1,254,983.16. Además, 76 líneas no tenían importe calculable por valores vacíos o no numéricos en `unit_price` o `quantity`. | No se eliminaron líneas: se separaron en `details_dep` (2200 líneas con orden válida) y `huerfanas` (15 líneas sin orden), para revisarlas. Supuesto: un `discount` vacío o inválido se trató como 0. | Al volver a ejecutar: líneas 2215 = 2200 + 15 e importe 1,254,983.16 = 1,250,608.41 + 4,374.75. Descuadre de líneas 0 y de importe 0.0. El `assert` de conservación de líneas pasó sin error. |

#Transformación 2

##Limpieza de texto e identificadores

Se quitan los espacios al inicio y al final de todas las columnas y el ".0" final de los order_id de order_details. Se hace primero porque estos errores son los que causan la mayoría de las líneas huérfanas del Control 2.

In [31]:
orders_limpio = orders_dep.apply(lambda c: c.str.strip())
details_limpio = df_details.drop(columns="importe").apply(lambda c: c.str.strip())
details_limpio["order_id"] = details_limpio["order_id"].str.replace(r"\.0$", "", regex=True)

print("Celdas con espacios sobrantes en orders:", (orders_dep != orders_limpio).sum().sum())
print("order_id corregidos en order_details:", (df_details["order_id"] != details_limpio["order_id"]).sum())

Celdas con espacios sobrantes en orders: 75
order_id corregidos en order_details: 9


###Caracteres mal codificados

Como la reparación de encoding no recupera el texto original (avance 1), se eliminan directamente las secuencias encontradas: "Ã¼", "Ã±", "Ã©" y "�".

In [32]:
patron_raro = r"Ã[¼±©]|�"

antes = orders_limpio.apply(lambda c: c.str.contains(patron_raro, regex=True)).sum().sum()
orders_limpio = orders_limpio.apply(lambda c: c.str.replace(patron_raro, "", regex=True).str.strip())
despues = orders_limpio.apply(lambda c: c.str.contains("[ÃÂ�]", regex=True)).sum().sum()

print("Celdas con caracteres extraños antes:", antes)
print("Celdas con caracteres extraños después:", despues)

Celdas con caracteres extraños antes: 43
Celdas con caracteres extraños después: 0


In [33]:
cambiadas = orders_dep["ship_city"].str.contains("Ã|�", regex=True)
display(pd.DataFrame({"antes": orders_dep.loc[cambiadas, "ship_city"],
                      "después": orders_limpio.loc[cambiadas, "ship_city"]}))

,antes,después
346,Regg�io Emilia,Reggio Emilia
410,M�ünchen,München
127,LaÃ¼nder,Lander
118,Anc�horage,Anchorage
375,A�nchorage,Anchorage
17,SeÃ¼attle,Seattle
320,Rio de Janeir�o,Rio de Janeiro
536,TsawaÃ±ssen,Tsawassen
269,M�éxico D.F.,México D.F.


###Duplicados en order_details

El Control 1 solo se aplicó a orders. Con los order_id ya normalizados se eliminan las filas idénticas de order_details.

In [34]:
dup_details = details_limpio.duplicated().sum()
details_limpio = details_limpio.drop_duplicates()
con_producto = details_limpio[details_limpio["product_id"] != ""]

print("Filas idénticas eliminadas:", dup_details)
print("Filas que quedan:", len(details_limpio))
print("Repetidas por order_id + product_id:", con_producto.duplicated(subset=["order_id", "product_id"]).sum())

Filas idénticas eliminadas: 60
Filas que quedan: 2155
Repetidas por order_id + product_id: 0


### Integridad entre tablas

Se repite la separación del Control 2, ahora con los order_id normalizados.

In [35]:
con_orden = details_limpio["order_id"].isin(orders_limpio["order_id"])
huerfanas = details_limpio.loc[~con_orden].copy()
details_limpio = details_limpio.loc[con_orden].copy()

print("Líneas con pedido:", len(details_limpio))
print("Líneas huérfanas:", len(huerfanas), sorted(huerfanas["order_id"]))
print("Pedidos de orders sin líneas:", (~orders_limpio["order_id"].isin(details_limpio["order_id"])).sum())
print("Líneas:", len(df_details), "=", dup_details, "+", len(details_limpio), "+", len(huerfanas))
assert dup_details + len(details_limpio) + len(huerfanas) == len(df_details)

Líneas con pedido: 2149
Líneas huérfanas: 6 ['19473', '19475', '19484', '19550', '19614', 'A10363']
Pedidos de orders sin líneas: 0
Líneas: 2215 = 60 + 2149 + 6


**Conclusión.** Se quitaron los espacios sobrantes de 75 celdas de orders y se corrigieron
9 order_id de order_details. Las 43 celdas con caracteres mal codificados quedaron limpias
(por ejemplo "SeÃ¼attle" ahora es "Seattle") y se eliminaron las 60 filas idénticas de
order_details, sin que quedara ninguna clave order_id + product_id repetida. Con los
identificadores normalizados las líneas huérfanas bajaron de 15 a 6 (19473, 19475, 19484,
19550, 19614 y A10363) y los 830 pedidos tienen al menos una línea. Las 6 huérfanas se
guardan aparte y no se borran. Las cuentas cuadran: 2215 = 60 + 2149 + 6.

#Transformación 3

##Conversión de columnas numéricas

Se convierten a número unit_price, quantity, discount y freight. La función limpiar_numero quita los símbolos de moneda ($ y USD), cambia la coma por punto, divide entre 100 los valores con "%" y reemplaza las palabras "diez" y "cinco" (en discount, "cinco" se toma como 5 %).

Lo que no se puede convertir (N/A y free) queda como NaN. A free no se le pone precio 0 porque según el diccionario el precio debe ser mayor que 0.

In [36]:
palabras = {"diez": "10", "cinco": "5%"}

def limpiar_numero(serie):
  s = serie.str.strip().str.lower().replace(palabras)
  s = s.str.replace("$", "", regex=False).str.replace("usd", "", regex=False)
  s = s.str.replace(",", ".", regex=False).str.strip()
  porcentaje = s.str.endswith("%")
  numero = pd.to_numeric(s.str.replace("%", "", regex=False), errors="coerce")
  numero.loc[porcentaje] = numero.loc[porcentaje] / 100
  return numero

def convertir(df, columna):
  original = df[columna]
  df[columna] = limpiar_numero(original)
  no_convertidos = original[(original != "") & df[columna].isna()]
  print(f"--- {columna} ---")
  print("Convertidos:", df[columna].notna().sum())
  print("Vacíos:", (original == "").sum())
  print("Quedan NaN:", len(no_convertidos), no_convertidos.unique().tolist())
  print()

In [37]:
convertir(details_limpio, "unit_price")
convertir(details_limpio, "quantity")
convertir(details_limpio, "discount")
convertir(orders_limpio, "freight")

--- unit_price ---
Convertidos: 2110
Vacíos: 25
Quedan NaN: 14 ['N/A', 'free']

--- quantity ---
Convertidos: 2129
Vacíos: 20
Quedan NaN: 0 []

--- discount ---
Convertidos: 2119
Vacíos: 30
Quedan NaN: 0 []

--- freight ---
Convertidos: 805
Vacíos: 25
Quedan NaN: 0 []



###Valores fuera de rango

Antes de aplicar las reglas del diccionario se revisa a qué pedidos pertenecen los descuentos que no son 0, 0.05, 0.10, 0.15, 0.20 o 0.25.

In [38]:
desc = details_limpio["discount"]
descuentos_validos = [0, 0.05, 0.10, 0.15, 0.20, 0.25]
raros = details_limpio[desc.notna() & ~desc.round(2).isin(descuentos_validos)]
display(raros.groupby("discount")["order_id"].unique())

discount
-0.10    [10757, 10280, 10989]
 0.01                  [11077]
 0.02                  [11077]
 0.03                  [11077]
 0.04                  [11077]
 0.06                  [11077]
 1.50    [10740, 10823, 10518]
Name: order_id, dtype: object

Los descuentos de 0.01 a 0.06 son todos del pedido 11077, por lo que se toman como algo
propio de ese pedido y no como un error. Por eso la regla queda como un rango de 0 a 0.25 y
no como una lista cerrada. Los valores -0.1 y 1.5 sí son inválidos (1.5 sería un descuento
del 150 %).

In [39]:
q = details_limpio["quantity"]
f = orders_limpio["freight"]
mala_q = q.notna() & ((q <= 0) | (q > 130) | (q % 1 != 0))
malo_desc = (desc < 0) | (desc > 0.25)
malo_f = f < 0

print("quantity ≤ 0, > 130 o con decimales:", mala_q.sum(), sorted(q[mala_q].tolist()))
print("discount fuera de 0 a 0.25:", malo_desc.sum(), sorted(desc[malo_desc].tolist()))
print("freight < 0:", malo_f.sum(), sorted(f[malo_f].tolist()))
print("unit_price ≤ 0:", (details_limpio["unit_price"] <= 0).sum())

details_limpio.loc[mala_q, "quantity"] = pd.NA
details_limpio.loc[malo_desc, "discount"] = pd.NA
orders_limpio.loc[malo_f, "freight"] = pd.NA

quantity ≤ 0, > 130 o con decimales: 9 [-25.0, -24.0, -5.0, 0.0, 0.0, 2.5, 15.5, 42.5, 999.0]
discount fuera de 0 a 0.25: 6 [-0.1, -0.1, -0.1, 1.5, 1.5, 1.5]
freight < 0: 4 [-136.54, -24.69, -12.36, -10.22]
unit_price ≤ 0: 0


Estos valores quedan como NaN para no perder el resto de la información de la línea. Los NaN se tratan más adelante en los controles de calidad.

###Tipos de dato

Los enteros se pasan a Int64 (con mayúscula) porque este tipo sí permite valores vacíos.

In [40]:
for col in ["order_id", "product_id", "quantity"]:
  details_limpio[col] = pd.to_numeric(details_limpio[col].replace("", pd.NA)).astype("Int64")
for col in ["order_id", "employee_id", "ship_via"]:
  orders_limpio[col] = pd.to_numeric(orders_limpio[col].replace("", pd.NA)).astype("Int64")

print(details_limpio.dtypes)
print()
print(orders_limpio[["order_id", "employee_id", "ship_via", "freight"]].dtypes)

order_id        Int64
product_id      Int64
unit_price    float64
quantity        Int64
discount      float64
dtype: object

order_id         Int64
employee_id      Int64
ship_via         Int64
freight        float64
dtype: object


###Importe recalculado

Se usa otra vez la función importe del Control 2.

In [41]:
details_limpio["importe"] = importe(details_limpio)

print("Líneas sin importe calculable en el Control 2:", df_details["importe"].isna().sum())
print("Líneas sin importe calculable ahora:", details_limpio["importe"].isna().sum())
print("Importe total:", round(details_limpio["importe"].sum(), 2))

Líneas sin importe calculable en el Control 2: 76
Líneas sin importe calculable ahora: 68
Importe total: 1229549.01


**Conclusión.** Después de limpiar símbolos, comas y palabras, solo quedaron sin convertir
14 precios (8 N/A y 6 free); quantity, discount y freight se convirtieron completas. Por
estar fuera de rango quedaron como NaN 9 cantidades (-25, -24, -5, 0, 0, 2.5, 15.5, 42.5 y
999), 6 descuentos (-0.1 y 1.5) y 4 fletes negativos. Todas las columnas quedaron con su
tipo correcto. Las líneas sin importe calculable bajaron de 76 a 68 y el importe total quedó
en 1,229,549.01, menor que en el Control 2 porque ya no cuenta duplicados ni huérfanas.

#Transformación 4

##Estandarización de fechas

En el diagnóstico 19 fechas con barra o guion eran ambiguas. Antes de convertir se revisa si el separador tiene relación con el orden del día y el mes.

In [42]:
for sep in ["/", "-"]:
  total = dia_primero = mes_primero = 0
  for col in columnas_fecha:
    s = orders_limpio[col]
    m = s[s.str.fullmatch(r"\d{2}" + sep + r"\d{2}" + sep + r"\d{4}")]
    total += len(m)
    dia_primero += (m.str[:2].astype(int) > 12).sum()
    mes_primero += (m.str[3:5].astype(int) > 12).sum()
  print(f"Con '{sep}': {total} fechas | día primero: {dia_primero} | mes primero: {mes_primero}")

s = pd.concat([orders_limpio[c] for c in columnas_fecha])
m = s[s.str.fullmatch(r"\d{2}\.\d{2}\.\d{2}")]
print(f"Con '.': {len(m)} fechas | día primero: {(m.str[:2].astype(int) > 12).sum()} | mes primero: {(m.str[3:5].astype(int) > 12).sum()}")

Con '/': 30 fechas | día primero: 20 | mes primero: 0
Con '-': 26 fechas | día primero: 0 | mes primero: 17
Con '.': 24 fechas | día primero: 17 | mes primero: 0


Con barra (/) el día siempre va primero (20 casos y ninguno al revés) y con guion (-)
siempre va primero el mes (17 casos y ninguno al revés). Con el punto pasa lo mismo que con
la barra. Por eso las fechas con barra se leen como DD/MM/AAAA y las de guion como
MM-DD-AAAA, incluidas las 19 que eran ambiguas. Las de año con dos dígitos se toman como
19xx, ya que todos los pedidos son de 1996 a 1998.

In [43]:
meses = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}

formatos = [
    (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d"),
    (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y"),
    (r"\d{2}-\d{2}-\d{4}", "%m-%d-%Y"),
    (r"\d{2}\.\d{2}\.\d{2}", "%d.%m.%y"),
    (r"\d{8}", "%Y%m%d"),
    (r"\d{4}/\d{2}/\d{2}", "%Y/%m/%d"),
    (r"[A-Za-z]{3} \d{2}, \d{4}", "%b %d, %Y"),
]

def convertir_fecha(x):
  if x == "":
    return pd.NaT
  for patron_f, formato in formatos:
    if re.fullmatch(patron_f, x):
      return pd.to_datetime(x, format=formato)
  m = re.fullmatch(r"(\d{1,2}) de ([a-z]+) de (\d{4})", x.lower())
  if m:
    return pd.Timestamp(int(m.group(3)), meses[m.group(2)], int(m.group(1)))
  return pd.NaT

In [44]:
for col in columnas_fecha:
  original = orders_limpio[col]
  orders_limpio[col] = pd.to_datetime(original.map(convertir_fecha))
  sin_convertir = (original != "") & orders_limpio[col].isna()
  print(f"--- {col} ---")
  print("Convertidas:", orders_limpio[col].notna().sum())
  print("Vacías:", (original == "").sum())
  print("Sin convertir:", sin_convertir.sum())
  print()

--- order_date ---
Convertidas: 820
Vacías: 10
Sin convertir: 0



--- required_date ---
Convertidas: 820
Vacías: 10
Sin convertir: 0



--- shipped_date ---
Convertidas: 809
Vacías: 21
Sin convertir: 0



###Coherencia entre fechas

Se repite la revisión del avance 1, pero ahora con todas las fechas y no solo con las que estaban en AAAA-MM-DD.

In [45]:
pedido = orders_limpio["order_date"]
requerido = orders_limpio["required_date"]
enviado = orders_limpio["shipped_date"]

print("Rango de order_date:", pedido.min().date(), "a", pedido.max().date())
print("Rango de shipped_date:", enviado.min().date(), "a", enviado.max().date())
print("Enviado antes de la fecha del pedido:", (enviado < pedido).sum())
print("Requerido antes de la fecha del pedido:", (requerido < pedido).sum())
print()
print("Días entre order_date y required_date:")
display((requerido - pedido).dt.days.value_counts())

Rango de order_date: 1996-07-04 a 1998-05-06
Rango de shipped_date: 1996-07-10 a 1998-05-06
Enviado antes de la fecha del pedido: 0
Requerido antes de la fecha del pedido: 0

Días entre order_date y required_date:


28.0    684
14.0     68
42.0     59
Name: count, dtype: int64

**Conclusión.** Se convirtieron todas las fechas con valor (820 en order_date, 820 en
required_date y 809 en shipped_date) y ninguna quedó sin convertir; los vacíos quedaron como
NaT. Revisando todas las fechas, ningún pedido se envió ni se requirió antes de hacerse, y
todo está entre 1996 y 1998. La diferencia entre order_date y required_date solo da 14, 28 o
42 días; si la regla de la barra y el guion estuviera mal saldrían diferencias raras, así
que esto confirma que la decisión fue correcta.

#Transformación 5

##Estandarización de categorías

###Nombres de país

Se usa la misma lista paises_validos del avance 1. Primero se compara sin tener en cuenta mayúsculas y despues se aplica un diccionario para los errores de escritura, los nombres en español y las siglas.

In [46]:
equivalencias = {
    "germnay": "Germany", "germay": "Germany", "frnace": "France", "francia": "France",
    "méxico": "Mexico", "méxco": "Mexico", "finalnd": "Finland", "venezuala": "Venezuela",
    "itally": "Italy", "italia": "Italy", "canda": "Canada", "austira": "Austria",
    "brasil": "Brazil", "united states": "USA", "eeuu": "USA",
    "u.k.": "UK", "united kingdom": "UK",
}
validos_minus = {p.lower(): p for p in paises_validos}

def estandarizar_pais(x):
  clave = x.lower()
  if clave in validos_minus:
    return validos_minus[clave]
  return equivalencias.get(clave, x)

antes = orders_limpio["ship_country"]
orders_limpio["ship_country"] = antes.map(estandarizar_pais)
pais = orders_limpio["ship_country"]

print("Valores distintos antes:", antes.nunique(), "| después:", pais.nunique())
print("Filas cambiadas:", (antes != pais).sum())
print("Filas con país no estándar:", ((pais != "") & ~pais.isin(paises_validos)).sum())
assert pais[pais != ""].isin(paises_validos).all()

Valores distintos antes: 50 | después: 22
Filas cambiadas: 63
Filas con país no estándar: 0


###Mayúsculas y minúsculas

En ship_name, ship_city y ship_address se deja una sola forma para los valores que solo cambian en mayúsculas. Se escoge la forma que combina mayúsculas y minúsculas ("Ernst Handel" y no "ERNST HANDEL").

In [47]:
def forma_comun(serie):
  s = serie[serie != ""]
  mixtas = s[~s.str.isupper() & ~s.str.islower()]
  forma = mixtas.groupby(mixtas.str.lower()).agg(lambda x: x.value_counts().index[0])
  return serie.map(lambda x: forma.get(x.lower(), x))

def formas_distintas(serie):
  s = serie[serie != ""]
  return (s.groupby(s.str.lower()).nunique() > 1).sum()

for col in ["ship_name", "ship_city", "ship_address"]:
  antes = orders_limpio[col]
  orders_limpio[col] = forma_comun(antes)
  print(col, "-> antes:", formas_distintas(antes), "| después:", formas_distintas(orders_limpio[col]),
        "| filas cambiadas:", (antes != orders_limpio[col]).sum())

ship_name -> antes: 16 | después: 0 | filas cambiadas: 18


ship_city -> antes: 14 | después: 0 | filas cambiadas: 19
ship_address -> antes: 8 | después: 0 | filas cambiadas: 8


###Errores de escritura por cliente

Al revisar los resultados se encontraron errores que no salieron en el diagnóstico: palabras mal escritas ("Bose" o "Bise" en vez de "Boise", "Reiims" en vez de "Reims") y otros caracteres insertados ("#", "@@", "???", "¤"). En el avance 1 solo se buscaron "Ã", "Â" y "�".

En Northwind cada cliente tiene una sola dirección de envío, por eso se revisa cuántos clientes tienen más de una forma del mismo campo.

In [48]:
def variantes_por_cliente(df, col):
  datos = df[(df["customer_id"] != "") & (df[col] != "")]
  return (datos.groupby("customer_id")[col].nunique() > 1).sum()

for col in ["ship_name", "ship_address", "ship_city", "ship_postal_code"]:
  print(col, "-> clientes con más de una forma:", variantes_por_cliente(orders_limpio, col))

print()
display(orders_limpio.loc[orders_limpio["customer_id"] == "SAVEA", "ship_city"].value_counts())

ship_name -> clientes con más de una forma: 27
ship_address -> clientes con más de una forma: 33
ship_city -> clientes con más de una forma: 27
ship_postal_code -> clientes con más de una forma: 8



ship_city
Boise      26
Bo#ise      1
Bois@@e     1
Bise        1
Bose        1
Name: count, dtype: int64

Se reemplaza cada variante por la forma que más se repite en el cliente. Si hay empate no se cambia nada, porque no se sabe cuál es la correcta. Las filas sin customer_id se quedan igual.

In [49]:
def corregir_por_cliente(df, col):
  datos = df[(df["customer_id"] != "") & (df[col] != "")]
  moda = datos.groupby("customer_id")[col].agg(lambda s: s.mode()[0] if len(s.mode()) == 1 else None)
  correcto = df["customer_id"].map(moda)
  cambiar = correcto.notna() & (df[col] != "") & (df[col] != correcto)
  return df[col].where(~cambiar, correcto), cambiar.sum()

for col in ["ship_name", "ship_address", "ship_city", "ship_postal_code"]:
  orders_limpio[col], cambios = corregir_por_cliente(orders_limpio, col)
  print(col, "-> filas corregidas:", cambios, "| clientes con más de una forma ahora:",
        variantes_por_cliente(orders_limpio, col))

print()
display(orders_limpio.loc[orders_limpio["customer_id"] == "SAVEA", "ship_city"].value_counts())

ship_name -> filas corregidas: 32 | clientes con más de una forma ahora: 1
ship_address -> filas corregidas: 38 | clientes con más de una forma ahora: 0


ship_city -> filas corregidas: 36 | clientes con más de una forma ahora: 0


ship_postal_code -> filas corregidas: 8 | clientes con más de una forma ahora: 0



ship_city
Boise    30
Name: count, dtype: int64

In [50]:
datos = orders_limpio[(orders_limpio["customer_id"] != "") & (orders_limpio["ship_name"] != "")]
empate = datos.groupby("customer_id")["ship_name"].nunique()
display(datos.loc[datos["customer_id"].isin(empate[empate > 1].index), ["order_id", "customer_id", "ship_name"]])

,order_id,customer_id,ship_name
824,10268,GROSR,GROSSELLA-Restaurante
439,10785,GROSR,GROSELLA-Restaurante


**Conclusión.** ship_country pasó de 50 valores distintos a 22 (los 21 países más el
vacío) con 63 filas corregidas. Las diferencias de mayúsculas en ship_name, ship_city y
ship_address desaparecieron (18, 19 y 8 filas). Con la corrección por cliente se arreglaron
32 filas de ship_name, 38 de ship_address, 36 de ship_city y 8 de ship_postal_code, por
ejemplo las 4 formas mal escritas de "Boise" del cliente SAVEA. Solo quedó sin resolver el
cliente GROSR, que tiene "GROSELLA-Restaurante" y "GROSSELLA-Restaurante" una vez cada uno.
Estos errores de escritura no se habían contado en el diagnóstico del avance 1.

## Revisión de las transformaciones

Se vuelven a correr las revisiones del diagnóstico sobre las tablas transformadas.

In [51]:
cols_texto = ["customer_id", "ship_name", "ship_address", "ship_city", "ship_region", "ship_postal_code", "ship_country"]
texto_orders = orders_limpio[cols_texto]
otros = r"\?\?\?|@@|¤|%\$"

print("Celdas con espacios al inicio o al final:", (texto_orders != texto_orders.apply(lambda c: c.str.strip())).sum().sum())
print("Celdas con caracteres extraños:", texto_orders.apply(lambda c: c.str.contains("[ÃÂ�]", regex=True)).sum().sum())
print("Celdas con '???', '@@', '¤' o '%$':", texto_orders.apply(lambda c: c.str.contains(otros, regex=True)).sum().sum())
print("order_id repetidos en orders:", orders_limpio["order_id"].duplicated().sum())
print("Filas idénticas en order_details:", details_limpio.duplicated().sum())
print("Líneas sin pedido en orders:", (~details_limpio["order_id"].isin(orders_limpio["order_id"])).sum())
print("Países no estándar:", ((orders_limpio["ship_country"] != "") & ~orders_limpio["ship_country"].isin(paises_validos)).sum())

Celdas con espacios al inicio o al final: 0
Celdas con caracteres extraños: 0
Celdas con '???', '@@', '¤' o '%$': 0
order_id repetidos en orders: 0
Filas idénticas en order_details: 0
Líneas sin pedido en orders: 0
Países no estándar: 0


## Resumen de las transformaciones

| Transformación | Problema / riesgo | Corrección o decisión | Evidencia de nueva prueba |
|---|---|---|---|
| 1. Unicidad de clave (`orders`) | 865 filas pero 830 `order_id` únicos: 25 duplicados exactos y 10 pedidos repetidos con `ship_name` distinto. | Se quitaron los duplicados exactos y en los 10 casos se dejó la fila con más campos llenos (`orders_dep`). | 830 filas, 830 `order_id` únicos y 0 duplicados. El `assert` pasó. |
| 2. Limpieza de texto e identificadores | Espacios sobrantes, 9 `order_id` con espacios o ".0", 43 celdas con caracteres mal codificados y 60 filas repetidas en `order_details`. Por esto salían 15 líneas huérfanas. | `strip()` en todas las columnas, se quitó el ".0", se eliminaron "Ã¼", "Ã±", "Ã©" y "�" y se borraron las filas idénticas de `order_details`. | 0 celdas con caracteres extraños. Huérfanas de 15 a 6 y 0 pedidos sin líneas. 2215 = 60 + 2149 + 6 (el `assert` pasó). |
| 3. Conversión numérica y rangos | Valores con $, USD, coma decimal, "%" y palabras; cantidades ≤ 0 o de 999, descuentos de -0.1 y 1.5 y fletes negativos. | Función `limpiar_numero`. Lo que no se puede convertir y lo que está fuera de rango queda como NaN. Se conservan los descuentos de 0.01 a 0.06 del pedido 11077. | Solo quedan sin convertir los 14 precios N/A y free. Tipos Int64 y float64. Líneas sin importe de 76 a 68. |
| 4. Estandarización de fechas | 8 formatos distintos y 19 fechas ambiguas. | Función `convertir_fecha`. Barra = DD/MM/AAAA y guion = MM-DD-AAAA, según lo que muestran los datos. | 0 fechas sin convertir, 0 fechas incoherentes y diferencias de solo 14, 28 o 42 días. |
| 5. Estandarización de categorías | 28 variantes de país, mayúsculas distintas y errores de escritura en los datos de envío. | Diccionario de equivalencias para países, forma con mayúsculas y minúsculas y forma más repetida por `customer_id`. | 0 países no estándar. Solo 1 cliente (GROSR) con dos formas de `ship_name`, por empate. |

*Las cifras corresponden a la ejecución del notebook.*

#Controles de calidad

##Manejo de valores nulos

Después de las transformaciones quedan valores vacíos y NaN: los que ya venían en los archivos y los que se pusieron por estar fuera de rango o no poderse convertir. Se cuentan por columna para decidir qué hacer con cada uno.

In [52]:
def nulos(df):
  n = df.replace("", pd.NA).isna().sum()
  resumen = pd.DataFrame({"nulos": n, "porcentaje": (n / len(df) * 100).round(2)})
  return resumen[resumen["nulos"] > 0].sort_values("nulos", ascending=False)

print("details_limpio")
display(nulos(details_limpio))
print()
print("orders_limpio")
display(nulos(orders_limpio))

details_limpio


,nulos,porcentaje
importe,68,3.16
unit_price,39,1.81
discount,36,1.68
quantity,29,1.35
product_id,15,0.70



orders_limpio


,nulos,porcentaje
ship_region,507,61.08
freight,29,3.49
shipped_date,21,2.53
ship_postal_code,19,2.29
ship_city,15,1.81
customer_id,15,1.81
employee_id,12,1.45
ship_name,12,1.45
order_date,10,1.20
required_date,10,1.20


###Cliente y datos de envío

Cada ship_name corresponde a un solo cliente, y en la transformación 5 se vio que cada cliente tiene una sola dirección de envío. Por eso los customer_id vacíos se completan a partir de ship_name, y los campos de envío vacíos a partir del customer_id.

In [53]:
datos = orders_limpio[(orders_limpio["customer_id"] != "") & (orders_limpio["ship_name"] != "")]
print("ship_name con más de un cliente:", (datos.groupby("ship_name")["customer_id"].nunique() > 1).sum())

cliente_por_nombre = datos.groupby("ship_name")["customer_id"].first()
vacio = orders_limpio["customer_id"] == ""
orders_limpio.loc[vacio, "customer_id"] = orders_limpio.loc[vacio, "ship_name"].map(cliente_por_nombre).fillna("")
print("customer_id vacíos antes:", vacio.sum(), "| después:", (orders_limpio["customer_id"] == "").sum())

ship_name con más de un cliente: 0
customer_id vacíos antes: 15 | después: 1


In [54]:
for col in ["ship_name", "ship_address", "ship_city", "ship_region", "ship_postal_code", "ship_country"]:
  datos = orders_limpio[(orders_limpio["customer_id"] != "") & (orders_limpio[col] != "")]
  valor_cliente = datos.groupby("customer_id")[col].agg(lambda s: s.mode()[0])
  vacio = (orders_limpio[col] == "") & (orders_limpio["customer_id"] != "")
  nuevo = orders_limpio.loc[vacio, "customer_id"].map(valor_cliente)
  orders_limpio.loc[vacio, col] = nuevo.fillna("")
  print(col, "-> vacíos:", vacio.sum(), "| completados:", nuevo.notna().sum())

ship_name -> vacíos: 11 | completados: 11


ship_address -> vacíos: 0 | completados: 0
ship_city -> vacíos: 15 | completados: 15
ship_region -> vacíos: 507 | completados: 0
ship_postal_code -> vacíos: 19 | completados: 0


ship_country -> vacíos: 10 | completados: 10


En ship_region solo se completan los clientes que sí tienen región en otros pedidos. Los demás se dejan vacíos porque la región no aplica a todos los países.

###Precio unitario

Se revisa cuántos precios distintos tiene cada producto.

In [55]:
precios = details_limpio.dropna(subset=["unit_price", "product_id"]).groupby("product_id")["unit_price"].nunique()
print("Productos según el número de precios distintos:")
display(precios.value_counts())

Productos según el número de precios distintos:


unit_price
2    75
3     2
Name: count, dtype: int64

Casi todos los productos tienen dos precios (uno antes y otro después de un cambio de precio), así que un promedio no sirve. Se completa el precio con el del mismo producto en el pedido más cercano por order_id. Antes de aplicarlo se prueba la regla con los precios que sí se conocen, como si no estuvieran.

In [56]:
conocidos = details_limpio.dropna(subset=["product_id", "unit_price"])

def precio_cercano(producto, pedido, excluir=None):
  mismos = conocidos[(conocidos["product_id"] == producto) & (conocidos.index != excluir)]
  if mismos.empty:
    return float("nan")
  cerca = (mismos["order_id"] - pedido).abs().idxmin()
  return mismos.loc[cerca, "unit_price"]

aciertos = sum(precio_cercano(f["product_id"], f["order_id"], i) == f["unit_price"] for i, f in conocidos.iterrows())
print("Aciertos de la regla:", aciertos, "de", len(conocidos), f"({aciertos / len(conocidos) * 100:.1f} %)")

Aciertos de la regla: 2047 de 2096 (97.7 %)


In [57]:
falta = details_limpio["unit_price"].isna() & details_limpio["product_id"].notna()
details_limpio.loc[falta, "unit_price"] = [precio_cercano(f.product_id, f.order_id) for f in details_limpio[falta].itertuples()]

print("Precios completados:", falta.sum())
print("Precios que siguen vacíos:", details_limpio["unit_price"].isna().sum())

Precios completados: 38
Precios que siguen vacíos: 1


###Descuento

En el Control 2 se asumió que un descuento vacío era 0. Se prueba una regla distinta: si las otras líneas del mismo pedido tienen un solo descuento, se usa ese. Si no hay de dónde sacarlo, se deja 0 como en el Control 2.

In [58]:
con_desc = details_limpio.dropna(subset=["discount"])

def descuento_pedido(pedido, excluir=None):
  otros_desc = con_desc[(con_desc["order_id"] == pedido) & (con_desc.index != excluir)]["discount"]
  return otros_desc.iloc[0] if otros_desc.nunique() == 1 else None

prueba = [(descuento_pedido(f["order_id"], i), f["discount"]) for i, f in con_desc.iterrows()]
prueba = [(p, real) for p, real in prueba if p is not None]
aciertos = sum(p == real for p, real in prueba)
print("Aciertos con el descuento del pedido:", aciertos, "de", len(prueba), f"({aciertos / len(prueba) * 100:.1f} %)")
print(f"Aciertos si siempre se pone 0: {(con_desc['discount'] == 0).mean() * 100:.1f} %")

Aciertos con el descuento del pedido: 1452 de 1651 (87.9 %)
Aciertos si siempre se pone 0: 60.8 %


In [59]:
falta = details_limpio["discount"].isna()
valores = [descuento_pedido(f.order_id) for f in details_limpio[falta].itertuples()]
por_pedido = sum(v is not None for v in valores)
details_limpio.loc[falta, "discount"] = [0 if v is None else v for v in valores]

print("Descuentos vacíos:", falta.sum())
print("Completados con el descuento del pedido:", por_pedido)
print("Completados con 0:", len(valores) - por_pedido)

Descuentos vacíos: 36
Completados con el descuento del pedido: 24
Completados con 0: 12


###Valores que no se completan

Para product_id, quantity, employee_id, freight y las fechas no hay una fuente confiable dentro de los datos, así que se dejan como NaN. Se recalcula el importe con los precios y descuentos completados.

In [60]:
details_limpio["importe"] = importe(details_limpio)

print("Líneas sin importe calculable:", details_limpio["importe"].isna().sum())
print("Importe total:", round(details_limpio["importe"].sum(), 2))
print()
print("details_limpio")
display(nulos(details_limpio))
print("orders_limpio")
display(nulos(orders_limpio))

Líneas sin importe calculable: 30
Importe total: 1249945.73

details_limpio


,nulos,porcentaje
importe,30,1.40
quantity,29,1.35
product_id,15,0.70
unit_price,1,0.05


orders_limpio


,nulos,porcentaje
ship_region,507,61.08
freight,29,3.49
shipped_date,21,2.53
ship_postal_code,19,2.29
employee_id,12,1.45
required_date,10,1.20
order_date,10,1.20
customer_id,1,0.12
ship_name,1,0.12


**Conclusión.** Se completaron 14 de los 15 customer_id vacíos a partir de ship_name, y con
el cliente se completaron 11 ship_name, 15 ship_city y 10 ship_country. Los 19 vacíos de
ship_postal_code son todos del cliente HUNGO (Cork), que no tiene código postal en ningún
pedido, y los 507 de ship_region corresponden a clientes sin región, así que son vacíos
esperables. Se completaron 38 precios con la regla del pedido más cercano, que acertó en el
97.7 % de los precios conocidos, y los 36 descuentos vacíos (24 con el descuento del pedido,
que acertó en el 87.9 %, y 12 con 0). Las líneas sin importe calculable bajaron de 68 a 30 y
el importe total quedó en 1,249,945.73. Siguen como NaN 29 cantidades, 15 product_id, 29
fletes, 12 employee_id y algunas fechas, porque no hay de dónde completarlos con seguridad.

#Integración de las fuentes

Se unen details_limpio y orders_limpio por order_id. La relación es de uno a muchos (un pedido tiene una o varias líneas), por eso se usa merge con validate="many_to_one": si hubiera un order_id repetido en orders, pandas mostraría un error en vez de duplicar líneas. Se usa how="left" desde details para no perder ninguna línea.

In [61]:
consolidado = details_limpio.merge(orders_limpio, on="order_id", how="left",
                                   validate="many_to_one", indicator=True)

print("Líneas en details_limpio:", len(details_limpio))
print("Líneas en el consolidado:", len(consolidado))
display(consolidado["_merge"].value_counts())
assert len(consolidado) == len(details_limpio)
consolidado = consolidado.drop(columns="_merge")

Líneas en details_limpio: 2149
Líneas en el consolidado: 2149


_merge
both          2149
left_only        0
right_only       0
Name: count, dtype: int64

##Validación del resultado

Se revisa que no se hayan perdido ni duplicado líneas o pedidos y que el importe total sea el mismo antes y después de unir.

In [62]:
print("Columnas:", consolidado.shape[1])
print("Pedidos distintos:", consolidado["order_id"].nunique(), "de", len(orders_limpio))
print("Repetidas por order_id + product_id:",
      consolidado.dropna(subset=["product_id"]).duplicated(subset=["order_id", "product_id"]).sum())
print("Importe antes de unir:", round(details_limpio["importe"].sum(), 2))
print("Importe después de unir:", round(consolidado["importe"].sum(), 2))
print("Líneas sin customer_id:", (consolidado["customer_id"] == "").sum())
assert consolidado["order_id"].nunique() == len(orders_limpio)
assert round(consolidado["importe"].sum(), 2) == round(details_limpio["importe"].sum(), 2)

Columnas: 19
Pedidos distintos: 830 de 830
Repetidas por order_id + product_id: 0
Importe antes de unir: 1249945.73
Importe después de unir: 1249945.73
Líneas sin customer_id: 3


#Dataset consolidado

Se guarda el resultado en northwind_consolidado.csv. Las 6 líneas huérfanas se guardan aparte en northwind_huerfanas.csv para que no se pierdan. Después se vuelve a leer el archivo para comprobar que se guardó completo.

In [63]:
consolidado.to_csv("northwind_consolidado.csv", index=False)
huerfanas.to_csv("northwind_huerfanas.csv", index=False)

revision = pd.read_csv("northwind_consolidado.csv")
print("northwind_consolidado.csv:", revision.shape)
print("northwind_huerfanas.csv:", pd.read_csv("northwind_huerfanas.csv").shape)
display(consolidado.head())

northwind_consolidado.csv: (2149, 19)
northwind_huerfanas.csv: (6, 5)


,order_id,product_id,unit_price,quantity,discount,importe,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
0,10474,40,14.7,21,0.00,308.7,PERIC,5,1997-03-13,1997-04-10,1997-03-21,2,83.49,Pericles Comidas clásicas,Calle Dr. Jorge Cash 321,México D.F.,,05033,Mexico
1,10527,36,19.0,30,0.10,513.0,QUICK,7,1997-05-05,1997-06-02,1997-05-07,1,41.90,QUICK-Stop,Taucherstraße 10,Cunewalde,,01307,Germany
2,10731,51,53.0,30,0.05,1510.5,CHOPS,7,1997-11-06,1997-12-04,1997-11-14,1,96.65,Chop-suey Chinese,Hauptstr. 31,Bern,,3012,Switzerland
3,11009,24,4.5,12,0.00,54.0,GODOS,2,1998-04-08,1998-05-06,1998-04-10,1,59.11,Godos Cocina Típica,"C/ Romero, 33",Sevilla,,41101,Spain
4,10657,60,34.0,30,0.00,1020.0,SAVEA,2,1997-09-04,1997-10-02,1997-09-15,2,352.69,Save-a-lot Markets,187 Suffolk Ln.,Boise,ID,83720,USA


**Conclusión.** La unión dejó las mismas 2149 líneas de details_limpio, todas con su
pedido (2149 "both" y ninguna "left_only"), y aparecen los 830 pedidos de orders. No hay
claves order_id + product_id repetidas y el importe total es igual antes y después de unir
(1,249,945.73), por lo que no se perdieron ni se duplicaron registros. Solo 3 líneas quedan
sin customer_id, todas del pedido 10498, que no tenía ni cliente ni ship_name. El resultado
se guardó en northwind_consolidado.csv (2149 filas y 19 columnas) y las 6 líneas huérfanas
en northwind_huerfanas.csv.